# Install:

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'gradio',
    'langchain==0.2.16',
    'langchain-community==0.2.16',
    'langchain-core==0.2.40',
    'accelerate',
    'bitsandbytes'
], check=True)

print('✅ Done')

# Load model:

In [1]:
import torch, re, threading
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer, pipeline as hf_pipeline

MODEL_NAME = 'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 4096,
    dtype          = None,
    load_in_4bit   = True,
    device_map     = 'auto',
)
FastLanguageModel.for_inference(model)

print('✅ Model loaded')
print(f'📊 Footprint: {model.get_memory_footprint()/1e9:.2f} GB')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

✅ Model loaded
📊 Footprint: 10.90 GB


# LangChain Tools:

In [2]:
from langchain.tools import tool

# Global pantry state
pantry: dict[str, int] = {}

# Generation helpers
def generate(prompt: str, max_new_tokens: int = 1024) -> str:
    inputs = tokenizer(text=prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def stream_tokens(prompt: str, max_new_tokens: int = 1024):
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    inputs   = tokenizer(text=prompt, return_tensors='pt').to(model.device)
    kwargs   = dict(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
        streamer       = streamer,
    )
    t = threading.Thread(target=model.generate, kwargs=kwargs, daemon=True)
    t.start()
    for token in streamer:
        yield token
    t.join()

def fmt_pantry() -> str:
    if not pantry:
        return 'Pantry is empty.'
    return 'Pantry: ' + ', '.join(f'{k} (x{v})' for k, v in sorted(pantry.items()))

_STOP = {'i','have','got','a','an','the','some','and','or','my','add','please','fresh','dried'}

def extract_ingredients(text: str) -> list[str]:
    text = re.sub(r'\b(i have|i got|add|remove|bought|picked up|i\'ve got)\b', '', text, flags=re.I)
    parts = re.split(r'[,;]|\band\b', text, flags=re.I)
    out = []
    for p in parts:
        p = re.sub(r'\b(a|an|the|some|\d+)\b', '', p, flags=re.I).strip().lower()
        if p and len(p) > 1 and p not in _STOP:
            out.append(p)
    return out

# LangChain Tools

@tool
def add_ingredients(ingredients: str) -> str:
    """
    Add one or more ingredients to the user's pantry.
    Input: comma-separated ingredient names. Example: 'tomato, cheese, eggs'
    """
    items = [i.strip().lower() for i in ingredients.split(',') if i.strip()]
    if not items:
        return 'No ingredients found. Provide a comma-separated list.'
    for item in items:
        pantry[item] = pantry.get(item, 0) + 1
    return f'Added: {", ".join(items)}.\n{fmt_pantry()}'


@tool
def remove_ingredient(ingredient: str) -> str:
    """
    Remove a single ingredient from the pantry.
    Input: ingredient name. Example: 'cheese'
    """
    ingredient = ingredient.strip().lower()
    if ingredient in pantry:
        del pantry[ingredient]
        return f'Removed "{ingredient}".\n{fmt_pantry()}'
    return f'"{ingredient}" not found in pantry.\n{fmt_pantry()}'


@tool
def view_pantry(query: str = '') -> str:
    """
    Show all current ingredients in the pantry.
    No input needed.
    """
    return fmt_pantry()


@tool
def clear_pantry(confirm: str = '') -> str:
    """
    Clear all ingredients from the pantry.
    Input: pass 'yes' to confirm.
    """
    if confirm.strip().lower() in ('yes', 'y', 'confirm'):
        pantry.clear()
        return 'Pantry cleared!'
    return 'Pass "yes" to confirm clearing the pantry.'


@tool
def suggest_recipes(preferences: str = '') -> str:
    """
    Suggest 3-5 recipes based on current pantry ingredients.
    Input: optional preferences like 'vegan', 'quick 20 min', 'Italian'.
    """
    if not pantry:
        return 'Pantry is empty. Ask the user to add ingredients first.'
    ingredients = ', '.join(sorted(pantry.keys()))
    pref_line = f'Preferences: {preferences}' if preferences else ''
    prompt = gem_prompt(
        f'You are an expert chef.\n'
        f'Available ingredients: {ingredients}\n{pref_line}\n\n'
        f'Suggest 3-5 creative recipes the user can make right now.\n'
        f'For each recipe include:\n'
        f'- Name and brief description\n'
        f'- Which pantry ingredients it uses\n'
        f'- Any extra common staples needed (salt, oil, etc.)\n'
        f'- Estimated cooking time and difficulty\n\n'
        f'Use **bold** for recipe names.'
    )
    return generate(prompt, max_new_tokens=1024)


@tool
def get_recipe_steps(recipe_name: str) -> str:
    """
    Get full step-by-step cooking instructions for a specific recipe.
    Input: recipe name. Example: 'Garlic Butter Pasta'
    """
    ctx = ', '.join(sorted(pantry.keys())) if pantry else 'general ingredients'
    prompt = gem_prompt(
        f'You are a professional chef. Write a complete recipe for: {recipe_name}\n'
        f'Available pantry ingredients: {ctx}\n\n'
        f'## Ingredients (with measurements)\n'
        f'## Step-by-Step Instructions (numbered)\n'
        f'## Chef\'s Tips\n'
        f'## Serving Suggestions\n\n'
        f'Keep it clear and beginner-friendly.'
    )
    return generate(prompt, max_new_tokens=1536)


TOOLS = [add_ingredients, remove_ingredient, view_pantry,
         clear_pantry, suggest_recipes, get_recipe_steps]

print(f'✅ {len(TOOLS)} LangChain tools registered:')
for t in TOOLS:
    print(f'   • {t.name} — {t.description.strip()[:60]}')

✅ 6 LangChain tools registered:
   • add_ingredients — Add one or more ingredients to the user's pantry.
Input: com
   • remove_ingredient — Remove a single ingredient from the pantry.
Input: ingredien
   • view_pantry — Show all current ingredients in the pantry.
No input needed.
   • clear_pantry — Clear all ingredients from the pantry.
Input: pass 'yes' to 
   • suggest_recipes — Suggest 3-5 recipes based on current pantry ingredients.
Inp
   • get_recipe_steps — Get full step-by-step cooking instructions for a specific re


# LangChain Agent:

In [4]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate

# Wrap model in LangChain LLM
lc_llm = HuggingFacePipeline(pipeline=hf_pipeline(
    'text-generation',
    model            = model,
    tokenizer        = tokenizer,
    max_new_tokens   = 512,
    temperature      = 0.3,
    do_sample        = True,
    pad_token_id     = tokenizer.eos_token_id,
    return_full_text = False,
))

# ReAct prompt for Gemma
REACT_PROMPT = PromptTemplate.from_template("""\
You are a kitchen assistant that manages a pantry and suggests recipes.
Use only the tools below. Do not invent information.

Tools:
{tools}

Format (follow exactly):
Question: {input}
Thought: which tool should I use?
Action: one of [{tool_names}]
Action Input: input for that tool
Observation: tool result
Thought: do I need another tool?
Final Answer: friendly reply to the user

Begin!
Question: {input}
Thought:{agent_scratchpad}""")

agent = create_react_agent(lc_llm, TOOLS, REACT_PROMPT)
agent_executor = AgentExecutor(
    agent                 = agent,
    tools                 = TOOLS,
    verbose               = True,
    max_iterations        = 4,
    handle_parsing_errors = True,
)

# Intent router calls LangChain tools directly

_ADD_RE    = re.compile(r'\b(add|i have|i got|bought|picked up|i\'ve got|have some)\b', re.I)
_REMOVE_RE = re.compile(r'\b(remove|delete|used up|ran out|don\'t have)\b', re.I)
_VIEW_RE   = re.compile(r'\b(pantry|what do i have|show.*ingredient|my ingredient)\b', re.I)
_CLEAR_RE  = re.compile(r'\b(clear pantry|empty pantry|reset pantry)\b', re.I)
_RECIPE_RE = re.compile(r'\b(what can i (make|cook)|suggest|dinner|lunch|breakfast|what.*cook)\b', re.I)
_STEPS_RE  = re.compile(r'\b(how (to|do i) (make|cook)|steps for|recipe for|instructions for)\b', re.I)

def run_agent(message: str) -> str:
    """Route message to the right LangChain tool, fallback to ReAct agent."""
    msg = message.strip()

    if _CLEAR_RE.search(msg):
        return clear_pantry.invoke({'confirm': 'yes'})

    if _VIEW_RE.search(msg):
        return view_pantry.invoke({'query': ''})

    if _REMOVE_RE.search(msg):
        cleaned = re.sub(_REMOVE_RE, '', msg, flags=re.I).strip()
        items   = extract_ingredients(cleaned)
        results = []
        for item in items:
            results.append(remove_ingredient.invoke({'ingredient': item}))
        return results[-1] if results else remove_ingredient.invoke({'ingredient': cleaned})

    if _ADD_RE.search(msg):
        items = extract_ingredients(msg)
        if items:
            return add_ingredients.invoke({'ingredients': ', '.join(items)})

    if _STEPS_RE.search(msg):
        name = re.sub(
            r'\b(how (to|do i) (make|cook)|steps for|recipe for|instructions for)\b',
            '', msg, flags=re.I
        ).strip().strip('?')
        return get_recipe_steps.invoke({'recipe_name': name})

    if _RECIPE_RE.search(msg):
        prefs = re.sub(_RECIPE_RE, '', msg, flags=re.I).strip()
        return suggest_recipes.invoke({'preferences': prefs})

    # Fallback: full LangChain ReAct agent
    try:
        result = agent_executor.invoke({'input': msg})
        return result.get('output', 'I could not process that.')
    except Exception as e:
        return f'Sorry, I had trouble with that. Try rephrasing. ({type(e).__name__})'

print('✅ LangChain agent ready')

✅ LangChain agent ready


# Gradio UI:

In [6]:
import gradio as gr

def chat(message, history):
    """Gradio chat function — streams recipe responses, instant for pantry ops."""
    msg = message.strip()

    # Streaming intents (recipe generation)
    is_streaming = _STEPS_RE.search(msg) or (
        _RECIPE_RE.search(msg) and pantry
    )

    if is_streaming:
        # Build prompt directly and stream tokens
        if _STEPS_RE.search(msg):
            name = re.sub(
                r'\b(how (to|do i) (make|cook)|steps for|recipe for|instructions for)\b',
                '', msg, flags=re.I
            ).strip().strip('?')
            ctx = ', '.join(sorted(pantry.keys())) if pantry else 'general ingredients'
            prompt = gem_prompt(
                f'You are a professional chef. Write a complete recipe for: {name}\n'
                f'Available pantry ingredients: {ctx}\n\n'
                f'## Ingredients (with measurements)\n'
                f'## Step-by-Step Instructions (numbered)\n'
                f'## Tips\n## Serving Suggestions\nKeep it beginner-friendly.'
            )
            max_tok = 1536
        else:
            prefs  = re.sub(_RECIPE_RE, '', msg, flags=re.I).strip()
            prompt = gem_prompt(
                f'You are an expert chef.\n'
                f'Available: {", ".join(sorted(pantry.keys()))}\n'
                f'{"Preferences: " + prefs if prefs else ""}\n\n'
                f'Suggest 3-5 recipes with name, ingredients used, '
                f'extra staples needed, cooking time and difficulty.'
            )
            max_tok = 1024

        response = ''
        for token in stream_tokens(prompt, max_tok):
            response += token
            yield response

    else:
        # Pantry ops — instant response via LangChain tools
        result = run_agent(msg)
        yield result


demo = gr.ChatInterface(
    fn          = chat,
    title       = '🍳 Recipe AI Agent',
    description = (
        'Powered by **Gemma 4 E4B** + **LangChain Tools**\n\n'
        'Add ingredients → get recipe suggestions → cook something delicious!'
    ),
    examples    = [
        'I have chicken, garlic, lemon and rosemary',
        'Add pasta, tomatoes, basil and parmesan',
        'What can I make for dinner?',
        'How do I make garlic pasta?',
        'Show my pantry',
        'Remove the chicken',
        'Clear pantry',
    ],
    theme       = gr.themes.Soft(),
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ea54d920ff4dbae968.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
